# We are going to make visuals for both MH and MK compounds

In [38]:
import polars as pl
from pathlib import Path
import numpy as np
import pyboat
from pyboat import WAnalyzer
from pyboat import plotting
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm
import pandas as pd
from scipy.signal import savgol_filter
import pandas as pd
import seaborn as sns

from dataclasses import dataclass
from typing import Generator

May (most recent) dataset

In [44]:
may_dataset_path = Path("/home/derfelt/PartchLabFiles/Circadian "
                    "Rhythms/Live Study/enVision/enVision_pre"
                    "processed/2026-05-11._.PREPROCESSED.parquet")

may_detected_period = Path("/home/derfelt/PartchLabFiles/Circadian "
                       "Rhythms/Live Study/enVision/period_dete"
                       "ction_results/5-11_per-well_period_results.csv")

may_dataset = pl.scan_parquet(may_dataset_path).filter(pl.col("feature") == "CalcResultI")

may_tpoints = may_dataset.select("Timepoint").unique().sort(by="Timepoint").collect().to_series().to_numpy()

may_detected_period = pl.read_csv(may_detected_period)
# display(detected_period)

may_platemap = (
    pl.read_excel("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/platemaps/2026-05-11"
                  "Echo_transfer.xlsx", engine='openpyxl')
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             "sample_type"])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
).rename({"384_Well": "Well"})

may_platemap = may_detected_period.join(may_platemap, on="Well", how='left')

mk_wells = (may_platemap
            .filter(pl.col("MoleculeID").str.contains("MK-", literal=True))
            .select("Well").to_series().to_list())
mk_dmso_wells = (may_platemap
            .filter(pl.col("MoleculeID").str.contains("DMSO", literal=True))
            .select("Well").to_series().to_list())

dex_wells = (may_platemap
            .filter(pl.col("MoleculeID").str.contains("Dex_", literal=True))
            .select("Well").to_series().to_list())

print(len(mk_wells), len(mk_dmso_wells), len(dex_wells))

print(len(may_tpoints))
display(may_dataset.head(2).collect())
display(may_platemap)



84 36 18
206


Well,Timepoint,feature,mean,median,stddev,q0,q10,q25,q50,q75,q90,q100
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""A01""",30.0,"""CalcResultI""",15.122279,15.122279,null,15.17334,15.163128,15.14781,15.122279,15.096748,15.08143,15.071217
"""A01""",30.5,"""CalcResultI""",-4.79441,-4.79441,null,-4.740661,-4.751411,-4.767536,-4.79441,-4.821285,-4.837409,-4.848159


Well,group_label,detected_period,MoleculeID,strain,sample_type
str,str,f64,str,str,str
"""G06""","""0.25% DMSO | bmal1""",29.727273,"""0.25% DMSO""","""bmal1""","""dex"""
"""G07""","""0.25% DMSO | bmal1""",29.727273,"""0.25% DMSO""","""bmal1""","""dex"""
"""G05""","""0.25% DMSO | bmal1""",29.727273,"""0.25% DMSO""","""bmal1""","""dex"""
"""L05""","""0.25% DMSO | per2""",25.153846,"""0.25% DMSO""","""per2""","""dex"""
"""L06""","""0.25% DMSO | per2""",25.153846,"""0.25% DMSO""","""per2""","""dex"""
…,…,…,…,…,…
"""O21""","""nan | nan""",25.153846,null,null,null
"""I03""","""nan | nan""",46.714286,null,null,null
"""N17""","""nan | nan""",23.357143,null,null,null


April Dataset (4-01)

In [ ]:
april_dataset_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                          "Study/enVision/enVision_preprocessed/30h-window._.20"
                          "26-04-01._.PREPROCESSED.parquet")

april_dataset = pl.scan_parquet(april_dataset_path).filter(pl.col("feature") == "CalcResultI")

april_detected_period = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/"
                             "Live Study/enVision/period_detection_results/4-"
                             "01_per-well_period_results.csv")
april_detected_period = pl.read_csv(april_detected_period)

april_platemap = (
    pl.read_excel("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                  "Study/platemaps/2026-04-01Echo"
                  "_transfer.xlsx", engine='openpyxl')
    .filter(pl.col("dex") == 'dex')
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             ])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
).rename({"384_Well": "Well"})

april_platemap = april_detected_period.join(april_platemap, on="Well", how='left')

mh_wells = (april_platemap
            .filter(pl.col("MoleculeID").str.contains("MH-", literal=True))
            .select("Well")
            .to_series()
            .to_list())

mh_dmso_wells = (april_platemap
            .filter(pl.col("MoleculeID").str.contains("DMSO", literal=True))
            .select("Well")
            .to_series()
            .to_list())

display(april_platemap)
display(april_dataset.head(2).collect())
print(len(mh_wells), len(mh_dmso_wells))

Well,group_label,detected_period,MoleculeID,strain
str,str,f64,str,str
"""D17""","""0.1% DMSO | Bmal1""",33.571429,"""0.1% DMSO""","""Bmal1"""
"""D15""","""0.1% DMSO | Bmal1""",33.571429,"""0.1% DMSO""","""Bmal1"""
"""D05""","""0.01% DMSO | Bmal1""",33.571429,"""0.01% DMSO""","""Bmal1"""
"""G14""","""0.01% DMSO | Bmal1""",33.571429,"""0.01% DMSO""","""Bmal1"""
"""E05""","""0.01% DMSO | Bmal1""",33.571429,"""0.01% DMSO""","""Bmal1"""
…,…,…,…,…
"""K04""","""nan | nan""",26.111111,null,null
"""O04""","""nan | nan""",26.111111,null,null
"""K15""","""nan | nan""",26.111111,null,null


Well,Timepoint,feature,mean,median,stddev,q0,q10,q25,q50,q75,q90,q100
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""A01""",30.0,"""CalcResultI""",-215.092717,-215.092717,null,-215.092717,-215.092717,-215.092717,-215.092717,-215.092717,-215.092717,-215.092717
"""A01""",30.5,"""CalcResultI""",-124.756786,-124.756786,null,-124.756786,-124.756786,-124.756786,-124.756786,-124.756786,-124.756786,-124.756786


36 24


# wavelet analysis for both compounds

In [73]:
@dataclass
class WaveletResults:
    """Wavlet analysis results"""
    ridge_df: pl.DataFrame
    summary_df: pl.DataFrame
    periods: np.ndarray
    fourier_df: pl.DataFrame

def add_period_marks(ax, x_min, x_max, interval=12):
    marks = np.arange(np.ceil(x_min / interval) * interval, x_max, interval)
    # marks = np.arange(x_min, x_max, interval)
    for mark in marks:
        ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.set_xticks(marks)
    ax.set_xticklabels([f'{int(m)}h' for m in marks], fontsize=7)

def add_period_marks_vlines_only(ax, x_min, x_max, interval=12):
    """Add vertical period marks without changing tick labels — safe for imshow axes"""
    marks = np.arange(np.ceil(x_min / interval) * interval, x_max + interval, interval)
    for mark in marks:
        ax.axvline(mark, color='white', linewidth=0.5, linestyle=':', alpha=0.7)

def calculate_all_wavelets(
    dataset: pl.LazyFrame,
    platemap: pl.DataFrame,
    name_col: str,
    well_list: list[str], 
    well_col: str, 
    stat_val: str,
    timepoint_col: str,
    sampling_interval: float = 1.0,
    period_range: tuple[int, int] = (18,40),
    n_periods: int=200,
    ridge_thresh: int=0,
) -> WaveletResults:
    periods = np.linspace(period_range[0], period_range[1], n_periods)
    ridge_results = []
    fourier_results =[]
    summary_rows = []
    
    for well in tqdm(well_list, desc='Wells'):
        well_df = (dataset
                   .filter(pl.col(well_col) == well)
                   .sort(timepoint_col)
                   )
        
        signal = well_df.select(stat_val).collect().to_series().to_numpy()
        tpoints = well_df.select(timepoint_col).collect().to_series().to_numpy()
        group = platemap.filter(pl.col(well_col) == well)[name_col].item()
        
        # wavelet analysis
        wan = WAnalyzer(periods=periods, dt=sampling_interval, time_unit_label='h')
        
        # spectrum computation
        wan.compute_spectrum(signal, do_plot=False)
        
        ridge = wan.get_maxRidge(power_thresh=ridge_thresh)
        fourier_power = wan.get_averaged_spectrum()
        
        # ridge summaries
        ridge['time'] = tpoints
        ridge[well_col] = well
        ridge[name_col] = group
        ridge_results.extend(ridge.to_dict(orient='records'))
        
        # fourier
        for p, pw in zip(periods, fourier_power):
            fourier_results.append({
                well_col: well,
                name_col: group,
                'period': p,
                'power': pw
            })
        
        # summaries
        mid_idx = len(ridge) // 2
        summary_rows.append({
            well_col          : well,
            name_col          : group,
            'mean_period'     : ridge['periods'].mean(),
            'std_period'      : ridge['periods'].std(),
            'median_period'   : ridge['periods'].median(),
            'mean_amplitude'  : ridge['amplitude'].mean(),
            'std_amplitude'   : ridge['amplitude'].std(),
            'median_amplitude': ridge['amplitude'].median(),
            'mid_phase'       : ridge['phase'].iloc[mid_idx],
            'peak_fourier_period': periods[np.argmax(fourier_power)],
        })
        
    
    return WaveletResults(
        ridge_df=pl.DataFrame(ridge_results),
        fourier_df=pl.DataFrame(fourier_results),
        periods=periods,
        summary_df=pl.DataFrame(summary_rows)
    )
    
    
def plot_wavelets(
    wavelet_results: WaveletResults,
    dataset: pl.LazyFrame,
    well_list: list[str],
    well_col: str,
    name_col: str,
    stat_val: str,
    timepoint_col: str,
    win_len: int = 13,
    polyorder: int = 3,
) -> Generator[plt.Figure]:
    
    ridge_df = wavelet_results.ridge_df.to_pandas()
    fourier_df = wavelet_results.fourier_df.to_pandas()
    periods = wavelet_results.periods
    
    for well in tqdm(well_list, desc='well plots'):
        well_signal = (
            dataset
            .filter(pl.col(well_col) == well)
            .sort(timepoint_col)
            .select([timepoint_col, stat_val])
            .collect()
            .to_pandas()
        )
        tpoints = well_signal[timepoint_col].to_numpy()
        signal = well_signal[stat_val].to_numpy()
        
        smoothed = savgol_filter(signal, window_length=win_len, polyorder=polyorder)
        well_signal[stat_val] = smoothed
        
        well_ridge = ridge_df[ridge_df[well_col] == well]
        well_fourier = fourier_df[fourier_df[well_col] == well]
        group = well_ridge[name_col].iloc[0]
        
        # plots
        fig, axs = plt.subplots(3, 1, figsize=(14,12))
        
        # detrended signal plot
        
        sns.lineplot(data=well_signal, 
                     x=timepoint_col, 
                     y=stat_val,
                     color='royalblue', 
                     linewidth=1.5,
                     ax=axs[0])
        axs[0].axhline(0, color='gray', linewidth=0.5, linestyle='--')
        axs[0].set_xlabel('Time (hours)')
        axs[0].set_ylabel('Expression')
        axs[0].set_title('Detrended Signal')
        axs[0].set_xlim(tpoints[0], tpoints[-1])
        add_period_marks(axs[0], tpoints[0], tpoints[-1])
        
        # wavelet spectrum
        
        wan = WAnalyzer(periods=periods, 
                        dt=(tpoints[1] - tpoints[0]), 
                        time_unit_label='h')
        wan.compute_spectrum(signal, do_plot=False)
        extent = [tpoints[0], tpoints[-1], periods[-1], periods[0]]
        axs[1].imshow(np.abs(wan.transform), aspect='auto', extent=extent, 
                      cmap='viridis', origin='upper')
        axs[1].plot(well_ridge['time'], well_ridge['periods'],
                       color='cyan', linewidth=2, label='ridge')
        axs[1].set_xlabel('Time (hours)')
        axs[1].set_ylabel('Period (hours)')
        axs[1].set_title('Wavelet Power Spectrum')
        axs[1].legend(fontsize=8)
        axs[1].set_xlim(tpoints[0], tpoints[-1])  # ← force x limits explicitly
        axs[1].set_ylim(periods[-1], periods[0])  # ← force y limits explicitly
        add_period_marks(axs[1], tpoints[0], tpoints[-1])
        
        # fourier spectrum
        peak_period = well_fourier.loc[well_fourier['power'].idxmax(), 'period']
        axs[2].plot(well_fourier['period'], well_fourier['power'],
                    color='darkred', linewidth=1.5)
        axs[2].axvline(peak_period, color='red', linewidth=1.5, linestyle='--',
                        label=f'Peak: {peak_period:.1f}h')
        axs[2].set_xlabel('Period (hours)')
        axs[2].set_ylabel('Power')
        axs[2].set_title('Fourier Spectrum (time-averaged)')
        axs[2].legend(fontsize=8)
        axs[2].set_xlim(periods[0], periods[-1])
        # add_period_marks(axs[2], periods[0], periods[-1])
        
        fig.suptitle(f"Well: {well} - {group}", fontsize=11)
        fig.tight_layout()
        
        yield fig

In [74]:
sampling_interval = 0.5 # hours
period_range = (18,40)
n_periods = 200
ridge_thresh = 2
timepoint_col = "Timepoint"

stat_val = "mean"

well_col = "Well"
wells = may_dataset.select(well_col).unique().collect().sort("Well").to_series().to_list()
print(wells)

win_len=13
polyorder=3
periods = np.linspace(period_range[0], period_range[1],n_periods)

out_path = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/wavelet_analysis")

results = []

out_pdf = out_path / "5-29-presentation._.wavelet_analysis_visuals.pdf"

# mk
mk_results = calculate_all_wavelets(
    dataset=may_dataset,
    platemap=may_platemap,
    name_col="group_label",
    well_list=mk_wells,
    well_col="Well",
    stat_val="mean",
    timepoint_col="Timepoint",
    sampling_interval=0.5
)

# DMSOs for mk
mk_DMSO_results = calculate_all_wavelets(
    dataset=may_dataset,
    platemap=may_platemap,
    name_col="group_label",
    well_list=mk_dmso_wells,
    well_col="Well",
    stat_val="mean",
    timepoint_col="Timepoint",
    sampling_interval=0.5
)

# dex 
dex_results = calculate_all_wavelets(
    dataset=may_dataset,
    platemap=may_platemap,
    name_col="group_label",
    well_list=dex_wells,
    well_col="Well",
    stat_val="mean",
    timepoint_col="Timepoint",
    sampling_interval=0.5
)

# mh 
mh_results = calculate_all_wavelets(
    dataset=april_dataset,
    platemap=april_platemap,
    name_col="group_label",
    well_list=mh_wells,
    well_col="Well",
    stat_val="mean",
    timepoint_col="Timepoint",
    sampling_interval=0.5
)

# dmso for mh
mh_dmso_results = calculate_all_wavelets(
    dataset=april_dataset,
    platemap=april_platemap,
    name_col="group_label",
    well_list=mh_dmso_wells,
    well_col="Well",
    stat_val="mean",
    timepoint_col="Timepoint",
    sampling_interval=0.5
)

with PdfPages(out_pdf) as pdf:
    for fig in plot_wavelets(
        wavelet_results=mk_results,
        dataset=may_dataset,
        well_list=mk_wells,
        well_col='Well',
        name_col='group_label',
        stat_val='mean',
        timepoint_col='Timepoint'
    ):
        pdf.savefig(fig)
        plt.close(fig)
    
    for fig in plot_wavelets(
        wavelet_results=mk_DMSO_results,
        dataset=may_dataset,
        well_list=mk_dmso_wells,
        well_col='Well',
        name_col='group_label',
        stat_val='mean',
        timepoint_col='Timepoint'
    ):
        pdf.savefig(fig)
        plt.close(fig)
    
    for fig in plot_wavelets(
        wavelet_results=mh_results,
        dataset=april_dataset,
        well_list=mh_wells,
        well_col='Well',
        name_col='group_label',
        stat_val='mean',
        timepoint_col='Timepoint'
    ):
        pdf.savefig(fig)
        plt.close(fig)
    
    for fig in plot_wavelets(
        wavelet_results=mh_dmso_results,
        dataset=april_dataset,
        well_list=mh_dmso_wells,
        well_col='Well',
        name_col='group_label',
        stat_val='mean',
        timepoint_col='Timepoint'
    ):
        pdf.savefig(fig)
        plt.close(fig)
    
    for fig in plot_wavelets(
        wavelet_results=dex_results,
        dataset=may_dataset,
        well_list=dex_wells,
        well_col='Well',
        name_col='group_label',
        stat_val='mean',
        timepoint_col='Timepoint'
    ):
        pdf.savefig(fig)
        plt.close(fig)
    
    
        

['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18', 'B19', 'B20', 'B21', 'B22', 'B23', 'B24', 'C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'D16', 'D17', 'D18', 'D19', 'D20', 'D21', 'D22', 'D23', 'D24', 'E01', 'E02', 'E03', 'E04', 'E05', 'E06', 'E07', 'E08', 'E09', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20', 'E21', 'E22', 'E23', 'E24', 'F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F09', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19', 'F20', 'F21', 'F22', 'F23'

Wells:   0%|          | 0/84 [00:00<?, ?it/s]

well plots: 100%|██████████| 18/18 [00:04<00:00,  3.93it/s]


# Let's make plots for period, phase, and amplitude over time

we will be using plotly for this because i am not doing all that

In [102]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import matplotlib.colors as mcolors
from natsort import natsorted

def get_gray_shades(n: int) -> list[str]:
    """Returns n gray shades from dark to light as hex strings"""
    return [
        '#{:02x}{:02x}{:02x}'.format(v, v, v)
        for v in [int(40 + 180 * i / max(n-1, 1)) for i in range(n)]
    ]

all_res_ridges = pl.concat([mk_results.ridge_df,
                            mk_DMSO_results.ridge_df,
                            mh_results.ridge_df,
                            mh_dmso_results.ridge_df,
                            dex_results.ridge_df])

all_res_ridges = all_res_ridges.with_columns(
    (pl.col('phase') / (2 * np.pi) * pl.col("periods")).alias("phase_hours")
)

display(all_res_ridges)

time,periods,phase,amplitude,power,frequencies,Well,group_label,phase_hours
f64,f64,f64,f64,f64,f64,str,str,f64
30.0,25.628141,2.583137,431.849186,47.491488,0.03902,"""F17""","""MK-8692 1.65uM | bmal1""",10.536216
30.5,25.628141,2.70428,436.048075,48.419502,0.03902,"""F17""","""MK-8692 1.65uM | bmal1""",11.03034
31.0,25.517588,2.813281,441.263961,49.370896,0.039189,"""F17""","""MK-8692 1.65uM | bmal1""",11.425438
31.5,25.517588,2.934685,445.335218,50.286126,0.039189,"""F17""","""MK-8692 1.65uM | bmal1""",11.918492
32.0,25.517588,3.056012,449.296277,51.184649,0.039189,"""F17""","""MK-8692 1.65uM | bmal1""",12.411231
…,…,…,…,…,…,…,…,…
130.5,25.959799,2.885892,158.949916,3.561886,0.038521,"""I16""","""Dex_PhenoVue_6TP_per2 | per2""",11.923439
131.0,25.959799,3.007781,155.218569,3.396619,0.038521,"""I16""","""Dex_PhenoVue_6TP_per2 | per2""",12.42704
131.5,26.070352,3.107792,151.417902,3.246082,0.038358,"""I16""","""Dex_PhenoVue_6TP_per2 | per2""",12.894929


In [111]:
metrics = ['periods', 'amplitude', 'phase_hours', 'phase']
name_col = 'group_label'
timepoint_col = 'time'

out_path = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/wavelet_analysis")

out_html = out_path / "5-29-presentation._.wavelet_analysis_groups._.BMAL1.html"

replicate_agg = (
    all_res_ridges
    .group_by([name_col, timepoint_col])
    .agg([
        *[pl.col(m).mean().alias(f'{m}_mean') for m in metrics],
        *[pl.col(m).std().alias(f'{m}_std') for m in metrics],
    ])
    .sort([name_col, timepoint_col])
    .to_pandas()
)
replicate_agg = replicate_agg[replicate_agg[name_col].str.contains('bmal1|Bmal1')]

groups = sorted(replicate_agg[name_col].unique())

# print(groups)
# print(len(groups))

# colormaps

import re
def extract_dmso_concentration(s: str) -> float:
    match = re.search(r'(\d+\.?\d*)\s*%', s)
    return float(match.group(1)) if match else 0.0

dmso_groups     = sorted([g for g in groups if 'dmso' in g.lower()],
                         key=extract_dmso_concentration,
                         reverse=True)
non_dmso_groups = natsorted([g for g in groups if 'dmso' not in g.lower()])
gray_shades     = get_gray_shades(len(dmso_groups))
palette         = px.colors.qualitative.Plotly

print(dmso_groups)

color_map = {}
for group, gray in zip(dmso_groups, gray_shades):
    color_map[group] = gray
for i, group in enumerate(non_dmso_groups):
    color_map[group] = palette[i % len(palette)]


# making subplots

fig = make_subplots(
    rows=len(metrics), cols=1,
    shared_xaxes=False,
    subplot_titles=[m.capitalize() for m in metrics],
    vertical_spacing=0.05
)

# colors = px.colors.qualitative.Plotly

for i, group in enumerate(groups):
    # color = colors[i % len(colors)]
    color = color_map[group]
    color_rgba = f'rgba({int(color[1:3], 16)}, {int(color[3:5], 16)}, {int(color[5:7], 16)}'
    
    gdf = replicate_agg[replicate_agg[name_col] == group]
    
    for row_idx, metric in enumerate(metrics, start=1):
        mean_col = f'{metric}_mean'
        std_col = f'{metric}_std'
        
        show_legend = (row_idx == 1)
        
        # shading
        fig.add_trace(go.Scatter(
            x=pd.concat([gdf['time'], gdf['time'].iloc[::-1]]),
            y=pd.concat([
                gdf[mean_col] + gdf[std_col],
                (gdf[mean_col] - gdf[std_col]).iloc[::-1]
            ]),
            fill='toself',
            fillcolor=f'{color_rgba}, 0.15)',
            line=dict(color='rgba(255,255,255,0)'),
            showlegend=False,
            legendgroup=group,
            name=group,
            hoverinfo='skip',
        ), row=row_idx, col=1)
        
        # mean line
        fig.add_trace(go.Scatter(
            x=gdf['time'],
            y=gdf[mean_col],
            mode='lines',
            name=group,
            legendgroup=group,
            showlegend=show_legend,
            line=dict(color=color, width=2),
            hovertemplate=f'<b>{group}</b><br>Time: %{{x:.1f}}h<br>{metric}: %{{y:.3f}}<extra></extra>',
        ), row=row_idx, col=1)
        
fig.update_layout(
    # height=300 * len(metrics),
    height=2000,
    title='Wavelet Ridge Analysis — All Groups',
    hovermode='x unified',
    legend=dict(
        groupclick='toggleitem',   # click legend to show/hide individual groups
        x=1.02,
        y=1,
    ),
)

fig.update_xaxes(title_text='Time (hours)', row=len(metrics), col=1)
for row_idx, metric in enumerate(metrics, start=1):
    fig.update_yaxes(title_text=metric.capitalize(), row=row_idx, col=1)


fig.update_traces(visible='legendonly')
fig.write_html(out_html,
    include_plotlyjs=True,
    post_script="""
    // Override legend click to toggle ALL traces with same legendgroup
    var myPlot = document.getElementsByClassName('plotly-graph-div')[0];
    myPlot.on('plotly_legendclick', function(data) {
        var clickedGroup = data.data[data.curveNumber].legendgroup;
        var update = {};
        var indices = [];
        
        // find all traces with same legendgroup
        data.data.forEach(function(trace, i) {
            if (trace.legendgroup === clickedGroup) {
                indices.push(i);
            }
        });
        
        // check current visibility of clicked trace
        var currentVis = data.data[data.curveNumber].visible;
        var newVis = (currentVis === 'legendonly' || currentVis === false) ? true : 'legendonly';
        
        var visUpdate = indices.map(function() { return newVis; });
        Plotly.restyle(myPlot, {'visible': visUpdate}, indices);
        
        return false; // prevent default behavior
    });
    """)
fig.show()

['1% DMSO | Bmal1', '0.5% DMSO | Bmal1', '0.25% DMSO | Bmal1', '0.25% DMSO | bmal1', '0.125% DMSO | bmal1', '0.1% DMSO | Bmal1', '0.0625% DMSO | bmal1', '0.05% DMSO | Bmal1', '0.03125% DMSO | bmal1', '0.0156% DMSO | bmal1', '0.01% DMSO | Bmal1', '0.0078% DMSO | bmal1', '0.005% DMSO | Bmal1']
